# 02 — Modélisation Multimodale, Fusion et Prédictions Finales 🚀

**Data Challenge :** Chimiométrie 2027 (SFDS)
**Auteurs :** Abdelouahad ALLA & Marcos FUNEZ FAMOSO (UMons)

## 1. Imports et Dépendances

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import ChemometricPreprocessor, BlockScaler, clean_physical_outliers, extract_modalities
from models import PLSDA
from sopls import SOPLSClassifier

## 2. Chargement et Normalisation Multivoie (Block Scaling)

In [ ]:
df_cal_raw = pd.read_excel('../data/Calibration_set_challenge.xlsx')
df_pred_raw = pd.read_excel('../data/Prediction_set_challenge.xlsx')

df_cal = clean_physical_outliers(df_cal_raw, is_train=True)
df_pred = clean_physical_outliers(df_pred_raw, is_train=False)

cal_mod = extract_modalities(df_cal)
pred_mod = extract_modalities(df_pred)

# Prétraitement NIR
prep = ChemometricPreprocessor(apply_snv=True, sg_window=15, sg_poly=2, sg_deriv=1)
X_cal_nir = prep.fit_transform(cal_mod['X_nir'])
X_pred_nir = prep.transform(pred_mod['X_nir'])

# Block Scaling (Frobenius)
blocks_cal = {'nir': X_cal_nir, 'morpho': cal_mod['X_morpho'], 'color': cal_mod['X_color']}
blocks_pred = {'nir': X_pred_nir, 'morpho': pred_mod['X_morpho'], 'color': pred_mod['X_color']}

block_scaler = BlockScaler()
block_scaler.fit(blocks_cal)
X_cal_blocked = block_scaler.transform(blocks_cal)
X_pred_blocked = block_scaler.transform(blocks_pred)

y = cal_mod['y']
print(f"Jeu d'entraînement : {X_cal_blocked.shape} | Test : {X_pred_blocked.shape}")

## 3. Évaluation du Modèle Optimal (Soft-Voting Ensemble) en 5-Fold CV

In [ ]:
clf_rbf = SVC(kernel='rbf', C=50.0, gamma=0.05, probability=True, random_state=42)
clf_lin = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
clf_et = ExtraTreesClassifier(n_estimators=300, max_features='sqrt', random_state=42, n_jobs=-1)

ensemble = VotingClassifier(
    estimators=[('svm_rbf', clf_rbf), ('svm_lin', clf_lin), ('et', clf_et)],
    voting='soft',
    weights=[3, 2, 1]
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []
y_true_all, y_pred_all = [], []

for train_idx, val_idx in skf.split(X_cal_blocked, y):
    ensemble.fit(X_cal_blocked[train_idx], y[train_idx])
    preds = ensemble.predict(X_cal_blocked[val_idx])
    scores.append(accuracy_score(y[val_idx], preds))
    y_true_all.extend(y[val_idx])
    y_pred_all.extend(preds)

print(f"Accuracy Moyenne (5-Fold CV) : {np.mean(scores)*100:.2f} % (± {np.std(scores)*100:.2f} %)")

## 4. Matrice de Confusion et Rapport Détaillé par Variété

In [ ]:
classes = sorted(np.unique(y))
print(classification_report(y_true_all, y_pred_all, target_names=classes, digits=3))

cm = confusion_matrix(y_true_all, y_pred_all, labels=classes)
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title(f"Matrice de Confusion Finale (Accuracy = {np.mean(scores)*100:.2f} %)")
plt.xlabel("Variété Prédite")
plt.ylabel("Variété Réelle")
plt.tight_layout()
plt.show()

## 5. Prédictions Finales pour le Challenge (700 Pois Inconnus)

In [ ]:
ensemble.fit(X_cal_blocked, y)
predictions = ensemble.predict(X_pred_blocked)

sub_df = pd.read_excel('../data/My_contribution.xlsx')
sub_df['Variety'] = predictions
print("Aperçu des 10 premières prédictions :")
print(sub_df.head(10))

print("\nDistribution des 700 prédictions :")
print(sub_df['Variety'].value_counts())